# BERT Sentiment Classification for Kaggle

This notebook trains a BERT transformer model for tweet sentiment classification with handling for class imbalance.

**Instructions for Kaggle:**
1. Upload `train.csv` and `test.csv` to Kaggle Dataset
2. Enable GPU in Kaggle notebook settings (Settings → Accelerator → GPU)
3. Run all cells
4. Download the trained model and predictions

In [ ]:
# Install required packages (if not already installed)
!pip install transformers -q
!pip install accelerate -q

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification, AdamW, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set random seeds
torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## Configuration

In [ ]:
# Configuration
CONFIG = {
    'model_name': 'bert-base-uncased',
    'max_length': 128,
    'batch_size': 32,  # Increased for GPU
    'epochs': 3,
    'learning_rate': 2e-5,
    'test_size': 0.2,
    'random_state': 42
}

# File paths - UPDATE THESE for your Kaggle dataset
TRAIN_PATH = '/kaggle/input/your-dataset/train.csv'  # Update this path
TEST_PATH = '/kaggle/input/your-dataset/test.csv'    # Update this path

# Or use local paths for testing
# TRAIN_PATH = '../input/train.csv'
# TEST_PATH = '../input/test.csv'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Custom Dataset Class

In [ ]:
class TweetDataset(Dataset):
    """Custom Dataset for tweet sentiment classification"""
    
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

## Load and Prepare Data

In [ ]:
# Load training data
df = pd.read_csv(TRAIN_PATH)

print(f"Dataset shape: {df.shape}")
print(f"\nFirst few rows:")
print(df.head())

# Label mapping
label_map = {0: 'Bearish', 1: 'Bullish', 2: 'Neutral'}

# Analyze class distribution
print("\nClass distribution:")
for label, count in df['label'].value_counts().sort_index().items():
    print(f"  {label_map[label]}: {count} ({count/len(df)*100:.1f}%)")

In [ ]:
# Split data with stratification
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['text'].values,
    df['label'].values,
    test_size=CONFIG['test_size'],
    random_state=CONFIG['random_state'],
    stratify=df['label']
)

print(f"Train samples: {len(train_texts)}")
print(f"Validation samples: {len(val_texts)}")

# Compute class weights for handling imbalance
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels),
    y=train_labels
)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

print("\nClass weights:")
for label, weight in enumerate(class_weights):
    print(f"  {label_map[label]}: {weight:.3f}")

## Initialize Model and Tokenizer

In [ ]:
# Initialize tokenizer and model
tokenizer = BertTokenizer.from_pretrained(CONFIG['model_name'])
model = BertForSequenceClassification.from_pretrained(
    CONFIG['model_name'],
    num_labels=3
)
model.to(device)

print(f"Model loaded: {CONFIG['model_name']}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Create datasets and dataloaders
train_dataset = TweetDataset(train_texts, train_labels, tokenizer, CONFIG['max_length'])
val_dataset = TweetDataset(val_texts, val_labels, tokenizer, CONFIG['max_length'])

train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'])

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

## Training Setup

In [ ]:
# Setup optimizer and scheduler
optimizer = AdamW(model.parameters(), lr=CONFIG['learning_rate'])
total_steps = len(train_loader) * CONFIG['epochs']
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

# Loss function with class weights
loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)

# History tracking
history = {
    'train_loss': [],
    'val_loss': [],
    'val_accuracy': []
}

## Training Loop

In [ ]:
# Training function
def train_epoch(model, dataloader, optimizer, scheduler, loss_fn, device):
    model.train()
    total_loss = 0
    
    progress_bar = tqdm(dataloader, desc="Training")
    for batch in progress_bar:
        optimizer.zero_grad()
        
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(outputs.logits, labels)
        
        total_loss += loss.item()
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        
        progress_bar.set_postfix({'loss': loss.item()})
    
    return total_loss / len(dataloader)

# Evaluation function
def eval_model(model, dataloader, loss_fn, device):
    model.eval()
    total_loss = 0
    predictions = []
    true_labels = []
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = loss_fn(outputs.logits, labels)
            total_loss += loss.item()
            
            preds = torch.argmax(outputs.logits, dim=1)
            predictions.extend(preds.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())
    
    avg_loss = total_loss / len(dataloader)
    accuracy = accuracy_score(true_labels, predictions)
    
    return avg_loss, accuracy, predictions, true_labels

In [ ]:
# Train the model
print("Starting training...")
print("=" * 80)

for epoch in range(CONFIG['epochs']):
    print(f"\nEpoch {epoch + 1}/{CONFIG['epochs']}")
    print("-" * 80)
    
    # Train
    train_loss = train_epoch(model, train_loader, optimizer, scheduler, loss_fn, device)
    
    # Evaluate
    val_loss, val_accuracy, val_preds, val_labels = eval_model(model, val_loader, loss_fn, device)
    
    # Save history
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_accuracy'].append(val_accuracy)
    
    print(f"\nTrain Loss: {train_loss:.4f}")
    print(f"Val Loss: {val_loss:.4f}")
    print(f"Val Accuracy: {val_accuracy:.4f}")

print("\n" + "=" * 80)
print("Training complete!")

## Evaluation Metrics

In [ ]:
# Compute final metrics
_, _, final_preds, final_labels = eval_model(model, val_loader, loss_fn, device)

# Compute metrics
accuracy = accuracy_score(final_labels, final_preds)
precision, recall, f1, support = precision_recall_fscore_support(
    final_labels, final_preds, average=None, labels=[0, 1, 2]
)

print("=" * 80)
print("EVALUATION METRICS")
print("=" * 80)
print(f"\nOverall Accuracy: {accuracy:.4f}")

print("\nPer-Class Metrics:")
print(f"{'Class':<15} {'Precision':<12} {'Recall':<12} {'F1-Score':<12} {'Support':<10}")
print("-" * 80)
for i in range(3):
    print(f"{label_map[i]:<15} {precision[i]:<12.4f} {recall[i]:<12.4f} "
          f"{f1[i]:<12.4f} {support[i]:<10}")

# Classification report
print("\n" + "=" * 80)
print("CLASSIFICATION REPORT")
print("=" * 80)
print(classification_report(
    final_labels, 
    final_preds,
    target_names=[label_map[i] for i in range(3)]
))

## Visualizations

In [ ]:
# Plot confusion matrix
cm = confusion_matrix(final_labels, final_preds)

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm, 
    annot=True, 
    fmt='d', 
    cmap='Blues',
    xticklabels=[label_map[i] for i in range(3)],
    yticklabels=[label_map[i] for i in range(3)]
)
plt.title('Confusion Matrix', fontsize=16, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Plot training history
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

epochs = range(1, len(history['train_loss']) + 1)

# Loss plot
ax1.plot(epochs, history['train_loss'], 'b-o', label='Train Loss')
ax1.plot(epochs, history['val_loss'], 'r-o', label='Val Loss')
ax1.set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy plot
ax2.plot(epochs, history['val_accuracy'], 'g-o', label='Val Accuracy')
ax2.set_title('Validation Accuracy', fontsize=14, fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Save Model

In [ ]:
# Save model and tokenizer
model.save_pretrained('./bert_sentiment_model')
tokenizer.save_pretrained('./bert_sentiment_model')

print("✓ Model saved to ./bert_sentiment_model")
print("\nDownload this folder from Kaggle to use the model locally!")

## Generate Predictions for Test Set (Optional)

In [ ]:
# Load test data and generate predictions
try:
    test_df = pd.read_csv(TEST_PATH)
    print(f"Test dataset loaded: {test_df.shape}")
    
    # Create test dataset (without labels)
    test_texts = test_df['text'].values
    test_encodings = [tokenizer.encode_plus(
        str(text),
        add_special_tokens=True,
        max_length=CONFIG['max_length'],
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors='pt'
    ) for text in tqdm(test_texts, desc="Tokenizing test data")]
    
    # Make predictions
    model.eval()
    test_predictions = []
    
    with torch.no_grad():
        for encoding in tqdm(test_encodings, desc="Predicting"):
            input_ids = encoding['input_ids'].to(device)
            attention_mask = encoding['attention_mask'].to(device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            pred = torch.argmax(outputs.logits, dim=1)
            test_predictions.append(pred.item())
    
    # Save predictions
    test_df['predicted_label'] = test_predictions
    test_df.to_csv('predictions.csv', index=False)
    
    print("✓ Predictions saved to predictions.csv")
    print(f"\nPrediction distribution:")
    for label, count in pd.Series(test_predictions).value_counts().sort_index().items():
        print(f"  {label_map[label]}: {count}")
        
except FileNotFoundError:
    print("Test file not found. Skipping predictions.")

## Summary

**Training Complete!**

Download the following from Kaggle:
1. `bert_sentiment_model/` - Trained model directory
2. `predictions.csv` - Test set predictions (if test data was provided)

**Next Steps:**
- Use the trained model for inference
- Fine-tune with different hyperparameters
- Integrate with multi-head ensemble (topics auxiliary task)

---

## One-Shot Training Script

Run everything in a single cell below (alternative to running all cells above):

In [ ]:
# ONE-SHOT FINBERT SENTIMENT CLASSIFICATION WITH DATA AUGMENTATION
# Complete training script in a single cell for Kaggle

# FIX: Reinstall PyTorch with compatible CUDA version for Kaggle GPU
print("Installing compatible PyTorch for Kaggle GPU...")
!pip uninstall torch torchvision torchaudio -y -q
!pip install torch==2.1.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 -q
print("✓ PyTorch reinstalled")

# Install transformers and accelerate
!pip install transformers accelerate -q
print("✓ Packages installed")

# Imports
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import BertTokenizer, BertForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set random seeds
torch.manual_seed(42)
np.random.seed(42)

# GPU Diagnostics
print("\n" + "="*80)
print("GPU DIAGNOSTICS")
print("="*80)
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"Device name: {torch.cuda.get_device_name(0)}")
    print(f"Device capability: {torch.cuda.get_device_capability(0)}")
    print(f"Current device: {torch.cuda.current_device()}")
    try:
        test_tensor = torch.tensor([1.0]).cuda()
        print(f"✓ CUDA test passed: {test_tensor.device}")
    except Exception as e:
        print(f"⚠ CUDA test failed: {e}")
else:
    print("⚠ WARNING: CUDA not available - will use CPU (very slow)")
print("="*80)

# Configuration
CONFIG = {
    'model_name': 'ProsusAI/finbert',  # ✨ Financial BERT
    'max_length': 128,
    'batch_size': 16,
    'epochs': 5,  # ✨ Increased to 5 epochs
    'learning_rate': 2e-5,
    'test_size': 0.2,
    'augment_prob': 0.15,  # ✨ Augmentation probability
}

# Paths
TRAIN_PATH = '/kaggle/input/datasets/franciscomiguel/text-mining-data/data/train.csv'
TEST_PATH = '/kaggle/input/datasets/franciscomiguel/text-mining-data/data/test.csv'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\n✓ Using device: {device}\n")

# ✨ GPU-Accelerated Data Augmentation
class GPUTextAugmenter:
    """GPU-accelerated text augmentation with token masking and deletion"""
    
    def __init__(self, mask_token_id, pad_token_id, vocab_size, augment_prob=0.15):
        self.mask_token_id = mask_token_id
        self.pad_token_id = pad_token_id
        self.vocab_size = vocab_size
        self.augment_prob = augment_prob
    
    def augment(self, input_ids, attention_mask):
        """Apply random token masking and deletion on GPU"""
        # 50% chance to skip augmentation
        if torch.rand(1).item() > 0.5:
            return input_ids, attention_mask
        
        # Clone to avoid modifying original
        aug_input_ids = input_ids.clone()
        aug_attention_mask = attention_mask.clone()
        
        # Handle both single samples (1D) and batches (2D)
        if aug_input_ids.dim() == 1:
            aug_input_ids = aug_input_ids.unsqueeze(0)
            aug_attention_mask = aug_attention_mask.unsqueeze(0)
            squeeze_output = True
        else:
            squeeze_output = False
        
        batch_size, seq_len = aug_input_ids.shape
        
        # Create mask for non-special tokens (not [CLS]=101, [SEP]=102, [PAD]=0)
        special_tokens_mask = (aug_input_ids == 101) | (aug_input_ids == 102) | (aug_input_ids == 0)
        augmentable_mask = ~special_tokens_mask & (aug_attention_mask == 1)
        
        # Random masking: replace tokens with [MASK] token
        mask_prob = torch.rand(batch_size, seq_len, device=aug_input_ids.device)
        should_mask = (mask_prob < self.augment_prob) & augmentable_mask
        aug_input_ids[should_mask] = self.mask_token_id
        
        # Random deletion: remove tokens by setting attention to 0
        delete_prob = torch.rand(batch_size, seq_len, device=aug_input_ids.device)
        should_delete = (delete_prob < self.augment_prob * 0.7) & augmentable_mask
        aug_attention_mask[should_delete] = 0
        
        # Squeeze back to 1D if input was 1D
        if squeeze_output:
            aug_input_ids = aug_input_ids.squeeze(0)
            aug_attention_mask = aug_attention_mask.squeeze(0)
        
        return aug_input_ids, aug_attention_mask

# ✨ Enhanced Dataset with GPU Augmentation
class TweetDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128, augmenter=None, is_train=False):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.augmenter = augmenter
        self.is_train = is_train
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        
        input_ids = encoding['input_ids'].flatten()
        attention_mask = encoding['attention_mask'].flatten()
        
        # Apply augmentation during training
        if self.is_train and self.augmenter is not None:
            input_ids, attention_mask = self.augmenter.augment(input_ids, attention_mask)
        
        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

# Load data
print("[1/6] Loading data...")
df = pd.read_csv(TRAIN_PATH)
label_map = {0: 'Bearish', 1: 'Bullish', 2: 'Neutral'}
print(f"Dataset: {len(df)} samples")
for label, count in df['label'].value_counts().sort_index().items():
    print(f"  {label_map[label]}: {count} ({count/len(df)*100:.1f}%)")

# Split data
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['text'].values, df['label'].values,
    test_size=CONFIG['test_size'], random_state=42, stratify=df['label']
)

# Class weights
class_weights = compute_class_weight(
    class_weight='balanced', classes=np.unique(train_labels), y=train_labels
)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)
print("\nClass weights:", {label_map[i]: f"{w:.3f}" for i, w in enumerate(class_weights)})

# Initialize model and tokenizer
print("\n[2/6] Initializing FinBERT model...")
tokenizer = BertTokenizer.from_pretrained(CONFIG['model_name'])
model = BertForSequenceClassification.from_pretrained(CONFIG['model_name'], num_labels=3)
model.to(device)
print(f"✓ Model loaded with {sum(p.numel() for p in model.parameters()):,} parameters")

# ✨ Initialize GPU augmenter
augmenter = GPUTextAugmenter(
    mask_token_id=tokenizer.mask_token_id,
    pad_token_id=tokenizer.pad_token_id,
    vocab_size=tokenizer.vocab_size,
    augment_prob=CONFIG['augment_prob']
)
print(f"✓ GPU Augmenter initialized (prob={CONFIG['augment_prob']})")

# Create datasets with augmentation
train_dataset = TweetDataset(train_texts, train_labels, tokenizer, CONFIG['max_length'], 
                             augmenter=augmenter, is_train=True)
val_dataset = TweetDataset(val_texts, val_labels, tokenizer, CONFIG['max_length'], 
                           augmenter=None, is_train=False)
train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'])

# Setup training with warmup
print("\n[3/6] Setting up training...")
optimizer = AdamW(model.parameters(), lr=CONFIG['learning_rate'])
total_steps = len(train_loader) * CONFIG['epochs']
num_warmup_steps = int(0.1 * total_steps)  # ✨ 10% warmup
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps, total_steps)
loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)
history = {'train_loss': [], 'val_loss': [], 'val_accuracy': []}
print(f"✓ Optimizer: AdamW (lr={CONFIG['learning_rate']})")
print(f"✓ Scheduler: Linear warmup ({num_warmup_steps} steps) + decay")
print(f"✓ Total training steps: {total_steps}")

# Training functions
def train_epoch(model, dataloader, optimizer, scheduler, loss_fn, device):
    model.train()
    total_loss = 0
    for batch in tqdm(dataloader, desc="Training"):
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(outputs.logits, labels)
        total_loss += loss.item()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
    return total_loss / len(dataloader)

def eval_model(model, dataloader, loss_fn, device):
    model.eval()
    total_loss = 0
    predictions, true_labels = [], []
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = loss_fn(outputs.logits, labels)
            total_loss += loss.item()
            preds = torch.argmax(outputs.logits, dim=1)
            predictions.extend(preds.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())
    return total_loss / len(dataloader), accuracy_score(true_labels, predictions), predictions, true_labels

# Train
print("\n[4/6] Training with augmentation...")
best_val_acc = 0
for epoch in range(CONFIG['epochs']):
    print(f"\nEpoch {epoch + 1}/{CONFIG['epochs']}")
    train_loss = train_epoch(model, train_loader, optimizer, scheduler, loss_fn, device)
    val_loss, val_accuracy, val_preds, val_labels = eval_model(model, val_loader, loss_fn, device)
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_accuracy'].append(val_accuracy)
    
    # Track best model
    if val_accuracy > best_val_acc:
        best_val_acc = val_accuracy
        best_epoch = epoch + 1
    
    print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_accuracy:.4f}")
    if val_accuracy == best_val_acc:
        print(f"✨ New best validation accuracy!")

print(f"\n✓ Best validation accuracy: {best_val_acc:.4f} (Epoch {best_epoch})")

# Evaluate
print("\n[5/6] Computing final metrics...")
_, _, final_preds, final_labels = eval_model(model, val_loader, loss_fn, device)
accuracy = accuracy_score(final_labels, final_preds)
precision, recall, f1, support = precision_recall_fscore_support(final_labels, final_preds, average=None, labels=[0,1,2])

print("\n" + "="*80)
print("EVALUATION METRICS")
print("="*80)
print(f"\nAccuracy: {accuracy:.4f}")
print(f"\n{'Class':<15} {'Precision':<12} {'Recall':<12} {'F1-Score':<12} {'Support':<10}")
print("-"*80)
for i in range(3):
    print(f"{label_map[i]:<15} {precision[i]:<12.4f} {recall[i]:<12.4f} {f1[i]:<12.4f} {support[i]:<10}")

print("\n" + classification_report(final_labels, final_preds, target_names=[label_map[i] for i in range(3)]))

# Visualizations
print("\n[6/6] Generating visualizations...")
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Confusion matrix
cm = confusion_matrix(final_labels, final_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=[label_map[i] for i in range(3)],
            yticklabels=[label_map[i] for i in range(3)])
axes[0].set_title('Confusion Matrix', fontweight='bold')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

# Loss curves
epochs = range(1, len(history['train_loss']) + 1)
axes[1].plot(epochs, history['train_loss'], 'b-o', label='Train Loss')
axes[1].plot(epochs, history['val_loss'], 'r-o', label='Val Loss')
axes[1].set_title('Training and Validation Loss', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Accuracy curve
axes[2].plot(epochs, history['val_accuracy'], 'g-o')
axes[2].axhline(y=best_val_acc, color='r', linestyle='--', alpha=0.5, label=f'Best: {best_val_acc:.4f}')
axes[2].set_title('Validation Accuracy', fontweight='bold')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Accuracy')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Save model
print("\nSaving model...")
model.save_pretrained('./finbert_sentiment_model')
tokenizer.save_pretrained('./finbert_sentiment_model')
print("✓ Model saved to ./finbert_sentiment_model")

# Predict test set
try:
    test_df = pd.read_csv(TEST_PATH)
    print(f"\nPredicting on test set ({len(test_df)} samples)...")
    test_dataset = TweetDataset(test_df['text'].values, [0]*len(test_df), tokenizer, CONFIG['max_length'])
    test_loader = DataLoader(test_dataset, batch_size=CONFIG['batch_size'])
    model.eval()
    test_preds = []
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Predicting"):
            outputs = model(batch['input_ids'].to(device), batch['attention_mask'].to(device))
            preds = torch.argmax(outputs.logits, dim=1)
            test_preds.extend(preds.cpu().numpy())
    test_df['predicted_label'] = test_preds
    test_df.to_csv('predictions.csv', index=False)
    print("✓ Predictions saved to predictions.csv")
    print("\nPrediction distribution:")
    for label, count in pd.Series(test_preds).value_counts().sort_index().items():
        print(f"  {label_map[label]}: {count}")
except:
    print("\nTest file not found. Skipping predictions.")

print("\n" + "="*80)
print("✓ TRAINING COMPLETE!")
print("="*80)
print(f"\n🎯 Final Results:")
print(f"   Model: FinBERT (financial domain)")
print(f"   Epochs: {CONFIG['epochs']}")
print(f"   Augmentation: GPU-accelerated token masking & deletion")
print(f"   Best Val Accuracy: {best_val_acc:.4f}")
print(f"   Final Accuracy: {accuracy:.4f}")
print("="*80)

---

## Vocabulary Analysis Agent

Advanced analysis to identify entities for replacement/normalization:
- Stock ticker symbols ($TSLA, AAPL)
- Company names (Tesla, Apple)
- Numerical patterns (prices, percentages)
- Social entities (mentions, URLs)
- Sentiment distribution analysis


In [ ]:
# RUN VOCABULARY ANALYSIS AGENT
# This analyzes the corpus to identify normalization patterns

import sys
sys.path.append('../preprocessing')

from vocabulary_analyzer import VocabularyAnalyzer

# Initialize analyzer with training data
analyzer = VocabularyAnalyzer(TRAIN_PATH)

# Run complete analysis pipeline
results = analyzer.run_full_analysis()

print("\n" + "="*80)
print("📊 ANALYSIS SUMMARY")
print("="*80)
print(f"\n🎯 Ticker Symbols Identified: {len(results['tickers'])}")
print(f"   High confidence (from $TICKER): {sum(1 for t in results['tickers'].values() if t['confidence'] == 'high')}")
print(f"   Medium confidence (from context): {sum(1 for t in results['tickers'].values() if t['confidence'] == 'medium')}")

print(f"\n🏢 Company Names Found: {len(results['companies'])}")
print(f"   Total mentions: {sum(results['companies'].values())}")

print(f"\n📈 Numerical Patterns:")
print(f"   Percentages: {results['numerical']['percentages']['total_occurrences']} occurrences")
print(f"   Prices: {results['numerical']['prices']['total_occurrences']} occurrences")
print(f"   Dates: {results['numerical']['dates']['total_occurrences']} occurrences")

print(f"\n👥 Social Entities:")
print(f"   User mentions: {results['social']['mentions']['total_occurrences']}")
print(f"   Hashtags: {results['social']['hashtags']['total_occurrences']}")
print(f"   URLs: {results['social']['urls']['total_occurrences']}")

print("\n" + "="*80)
print("💡 REPLACEMENT STRATEGY")
print("="*80)
strategy = results['replacement_strategy']
print("\nPriority Order:")
for priority in strategy['priority_order']:
    print(f"  {priority}")

print(f"\n✨ Expected Improvement: {strategy['estimated_total_impact']['expected_accuracy_gain']}")
print(f"   Vocabulary reduction: {strategy['estimated_total_impact']['vocabulary_reduction']}")
print(f"   Primary benefit: {strategy['estimated_total_impact']['primary_benefit']}")

print("\n" + "="*80)
print("📁 Output Files Generated:")
print("="*80)
print("  ✓ ./outputs/vocab_analysis/vocabulary_analysis.json (complete results)")
print("  ✓ ./outputs/vocab_analysis/tickers_to_replace.json (ticker list)")
print("  ✓ ./outputs/vocab_analysis/companies_to_replace.json (company list)")
print("  ✓ ./outputs/vocab_analysis/replacement_strategy.md (strategy report)")
print("  ✓ ./outputs/vocab_analysis/vocabulary_analysis.png (visualizations)")
print("="*80)


### Top Tickers Analysis

Let's look at the most frequently mentioned tickers and their sentiment distribution:


In [ ]:
# Display top 20 tickers with their statistics
print("TOP 20 TICKER SYMBOLS")
print("="*80)
print(f"{'Rank':<6} {'Ticker':<10} {'Mentions':<12} {'Confidence':<15} {'Source':<12}")
print("-"*80)

for rank, (ticker, info) in enumerate(list(results['tickers'].items())[:20], 1):
    print(f"{rank:<6} ${ticker:<9} {info['count']:<12} {info['confidence']:<15} {info['source']:<12}")

# Display top companies
print("\n\nTOP 15 COMPANY NAMES")
print("="*80)
print(f"{'Rank':<6} {'Company':<30} {'Mentions':<12}")
print("-"*80)

for rank, (company, count) in enumerate(list(results['companies'].items())[:15], 1):
    print(f"{rank:<6} {company:<30} {count:<12}")

# Display ticker sentiment analysis
if 'sentiment_analysis' in results and results['sentiment_analysis']['ticker_sentiment']:
    print("\n\nTICKER SENTIMENT DISTRIBUTION (Top 10)")
    print("="*80)
    print(f"{'Ticker':<10} {'Bearish':<12} {'Neutral':<12} {'Bullish':<12} {'Total':<12} {'Dominant':<12}")
    print("-"*80)
    
    for ticker, sentiment in list(results['sentiment_analysis']['ticker_sentiment'].items())[:10]:
        total = sum(sentiment.values())
        dominant = max(sentiment, key=sentiment.get)
        print(f"${ticker:<9} {sentiment['Bearish']:<12} {sentiment['Neutral']:<12} {sentiment['Bullish']:<12} {total:<12} {dominant:<12}")


---

## Text Normalization (Offline Preprocessing)

Apply deterministic entity replacement to reduce vocabulary sparsity:
- Tickers: `$TSLA` → `[TICKER]`
- Companies: `Tesla` → `[COMPANY]`
- Mentions: `@elonmusk` → `[USER]`
- URLs: `https://...` → `[URL]`

**Expected Impact:** +3-6% accuracy improvement

In [ ]:
# ============================================================================
# ENHANCED TRAINING PIPELINE - ONE-SHOT CELL
# Extended Pretraining + Hard GPU Augmentation + Normalization
# Target: 89-93% accuracy
# ============================================================================

import sys
import os
sys.path.insert(0, os.path.abspath('../'))

import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import BertTokenizer, BertForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
from sklearn.utils.class_weight import compute_class_weight
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)

# ============================================================================
# HARDER GPU AUGMENTER
# ============================================================================
class HarderGPUAugmenter:
    """
    HARDER GPU-accelerated augmentation:
    - 20% masking (up from 15%)
    - 15% deletion (up from 10.5%)
    - 10% random replacement (NEW!)
    - No 50% skip chance (always augments)
    """
    
    def __init__(self, mask_token_id, pad_token_id, vocab_size, augment_prob=0.20):
        self.mask_token_id = mask_token_id
        self.pad_token_id = pad_token_id
        self.vocab_size = vocab_size
        self.augment_prob = augment_prob
    
    def augment(self, input_ids, attention_mask):
        aug_input_ids = input_ids.clone()
        aug_attention_mask = attention_mask.clone()
        
        if aug_input_ids.dim() == 1:
            aug_input_ids = aug_input_ids.unsqueeze(0)
            aug_attention_mask = aug_attention_mask.unsqueeze(0)
            squeeze_output = True
        else:
            squeeze_output = False
        
        batch_size, seq_len = aug_input_ids.shape
        
        # Mask for non-special tokens
        special_tokens_mask = (aug_input_ids == 101) | (aug_input_ids == 102) | (aug_input_ids == 0)
        augmentable_mask = ~special_tokens_mask & (aug_attention_mask == 1)
        
        # 1. Random masking: 20%
        mask_prob = torch.rand(batch_size, seq_len, device=aug_input_ids.device)
        should_mask = (mask_prob < self.augment_prob) & augmentable_mask
        aug_input_ids[should_mask] = self.mask_token_id
        
        # 2. Random deletion: 15%
        delete_prob = torch.rand(batch_size, seq_len, device=aug_input_ids.device)
        should_delete = (delete_prob < self.augment_prob * 0.75) & augmentable_mask & ~should_mask
        aug_attention_mask[should_delete] = 0
        
        # 3. Random replacement: 10% (NEW!)
        replace_prob = torch.rand(batch_size, seq_len, device=aug_input_ids.device)
        should_replace = (replace_prob < 0.10) & augmentable_mask & ~should_mask & ~should_delete
        random_tokens = torch.randint(
            low=1000,
            high=min(20000, self.vocab_size),
            size=(batch_size, seq_len),
            device=aug_input_ids.device
        )
        aug_input_ids[should_replace] = random_tokens[should_replace]
        
        if squeeze_output:
            aug_input_ids = aug_input_ids.squeeze(0)
            aug_attention_mask = aug_attention_mask.squeeze(0)
        
        return aug_input_ids, aug_attention_mask

# ============================================================================
# DATASET CLASS
# ============================================================================
class TweetDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128, augmenter=None, is_train=False):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.augmenter = augmenter
        self.is_train = is_train
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        
        input_ids = encoding['input_ids'].flatten()
        attention_mask = encoding['attention_mask'].flatten()
        
        if self.is_train and self.augmenter is not None:
            input_ids, attention_mask = self.augmenter.augment(input_ids, attention_mask)
        
        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

# ============================================================================
# TRAINING FUNCTIONS
# ============================================================================
def train_epoch(model, dataloader, optimizer, scheduler, loss_fn, device):
    model.train()
    total_loss = 0
    for batch in tqdm(dataloader, desc="Training"):
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(outputs.logits, labels)
        total_loss += loss.item()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
    return total_loss / len(dataloader)

def eval_model(model, dataloader, loss_fn, device):
    model.eval()
    total_loss = 0
    predictions, true_labels = [], []
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = loss_fn(outputs.logits, labels)
            total_loss += loss.item()
            preds = torch.argmax(outputs.logits, dim=1)
            predictions.extend(preds.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())
    return total_loss / len(dataloader), accuracy_score(true_labels, predictions), predictions, true_labels

# ============================================================================
# MAIN PIPELINE
# ============================================================================
print("="*80)
print("ENHANCED TRAINING PIPELINE")
print("Step 1: Extended Pretraining (MLM + Topics)")
print("Step 2: Fine-tuning with Hard Augmentation")
print("="*80)

# Configuration
CONFIG = {
    'pretrained_model': '../pretrained_finbert',
    'base_model': 'ProsusAI/finbert',
    'use_pretrained': True,  # Set to False to skip pretraining
    'max_length': 128,
    'batch_size': 16,
    'epochs': 7,
    'learning_rate': 2e-5,
    'test_size': 0.2,
    'augment_prob': 0.20,
}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\n✓ Using device: {device}\n")

# ============================================================================
# STEP 1: EXTENDED PRETRAINING
# ============================================================================
if CONFIG['use_pretrained']:
    print("\n[STEP 1] Checking for pretrained model...")
    if not os.path.exists(CONFIG['pretrained_model']):
        print("  ⚠ Pretrained model not found. Running pretraining first...")
        print("  This will take ~15-30 minutes on GPU\n")
        
        from preprocessing.extended_pretraining import run_pretraining
        
        df_full = pd.read_csv('../preprocessed_data/train_normalized.csv')
        run_pretraining(
            train_texts=df_full['text_normalized'].values,
            model_name=CONFIG['base_model'],
            output_path=CONFIG['pretrained_model'],
            num_epochs=3,
            batch_size=32,
            learning_rate=5e-5,
            mask_prob=0.30
        )
        print("\n  ✓ Pretraining complete!")
    else:
        print(f"  ✓ Found pretrained model at {CONFIG['pretrained_model']}")
    
    model_name = CONFIG['pretrained_model']
else:
    print("\n[STEP 1] Skipping pretraining, using base model")
    model_name = CONFIG['base_model']

# ============================================================================
# STEP 2: LOAD DATA
# ============================================================================
print("\n[STEP 2] Loading normalized data...")
df = pd.read_csv('../preprocessed_data/train_normalized.csv')
label_map = {0: 'Bearish', 1: 'Bullish', 2: 'Neutral'}
print(f"  Dataset: {len(df)} samples")
for label, count in df['label'].value_counts().sort_index().items():
    print(f"    {label_map[label]}: {count} ({count/len(df)*100:.1f}%)")

# Split data
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['text_normalized'].values,
    df['label'].values,
    test_size=CONFIG['test_size'],
    random_state=42,
    stratify=df['label']
)

# Class weights
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels),
    y=train_labels
)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)
print(f"\n  Class weights: {', '.join([f'{label_map[i]}: {w:.3f}' for i, w in enumerate(class_weights)])}")

# ============================================================================
# STEP 3: INITIALIZE MODEL
# ============================================================================
print("\n[STEP 3] Initializing model with harder augmentation...")
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertForSequenceClassification.from_pretrained(model_name, num_labels=3)
model.to(device)
print(f"  ✓ Model loaded: {sum(p.numel() for p in model.parameters()):,} parameters")

augmenter = HarderGPUAugmenter(
    mask_token_id=tokenizer.mask_token_id,
    pad_token_id=tokenizer.pad_token_id,
    vocab_size=tokenizer.vocab_size,
    augment_prob=CONFIG['augment_prob']
)
print(f"  ✓ HARDER GPU Augmenter: {CONFIG['augment_prob']*100}% prob + replacements")

# Create datasets
train_dataset = TweetDataset(train_texts, train_labels, tokenizer, CONFIG['max_length'],
                             augmenter=augmenter, is_train=True)
val_dataset = TweetDataset(val_texts, val_labels, tokenizer, CONFIG['max_length'],
                           augmenter=None, is_train=False)
train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'])

# ============================================================================
# STEP 4: TRAINING
# ============================================================================
print("\n[STEP 4] Training with hard augmentation...")
optimizer = AdamW(model.parameters(), lr=CONFIG['learning_rate'])
total_steps = len(train_loader) * CONFIG['epochs']
num_warmup_steps = int(0.1 * total_steps)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps, total_steps)
loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)
history = {'train_loss': [], 'val_loss': [], 'val_accuracy': []}

best_val_acc = 0
for epoch in range(CONFIG['epochs']):
    print(f"\nEpoch {epoch + 1}/{CONFIG['epochs']}")
    train_loss = train_epoch(model, train_loader, optimizer, scheduler, loss_fn, device)
    val_loss, val_accuracy, val_preds, val_labels = eval_model(model, val_loader, loss_fn, device)
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_accuracy'].append(val_accuracy)
    
    if val_accuracy > best_val_acc:
        best_val_acc = val_accuracy
        best_epoch = epoch + 1
        model.save_pretrained('../best_model')
        tokenizer.save_pretrained('../best_model')
    
    print(f"  Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_accuracy:.4f}")
    if val_accuracy == best_val_acc:
        print(f"  ✨ New best validation accuracy!")

print(f"\n✓ Best validation accuracy: {best_val_acc:.4f} (Epoch {best_epoch})")

# ============================================================================
# STEP 5: FINAL EVALUATION
# ============================================================================
print("\n[STEP 5] Final evaluation...")
_, _, final_preds, final_labels = eval_model(model, val_loader, loss_fn, device)
accuracy = accuracy_score(final_labels, final_preds)
precision, recall, f1, support = precision_recall_fscore_support(
    final_labels, final_preds, average=None, labels=[0,1,2]
)

print("\n" + "="*80)
print("FINAL RESULTS")
print("="*80)
print(f"\nAccuracy: {accuracy:.4f}")
print(f"\n{'Class':<15} {'Precision':<12} {'Recall':<12} {'F1-Score':<12} {'Support':<10}")
print("-"*80)
for i in range(3):
    print(f"{label_map[i]:<15} {precision[i]:<12.4f} {recall[i]:<12.4f} {f1[i]:<12.4f} {support[i]:<10}")

print("\n" + classification_report(final_labels, final_preds, target_names=[label_map[i] for i in range(3)]))

print("\n" + "="*80)
print("✓ TRAINING COMPLETE!")
print("="*80)
print(f"\n🎯 Results Summary:")
print(f"   Pretraining: {'Used' if CONFIG['use_pretrained'] else 'Skipped'}")
print(f"   Augmentation: HARDER (20% prob + replacements)")
print(f"   Normalization: Enabled (tickers, prices, percentages)")
print(f"   Best Val Accuracy: {best_val_acc:.4f}")
print(f"   Final Accuracy: {accuracy:.4f}")
print(f"\n💡 Baseline: 85.91% | With Normalization: 86.85%")
print(f"   Target: 89-93% | Achieved: {'✅ YES!' if accuracy >= 0.89 else '⏳ Getting there!'}")
print("="*80)

In [ ]:
# NORMALIZE TRAINING DATA
# This creates preprocessed versions of train/test datasets

import sys
sys.path.append('../preprocessing')

from text_normalizer import TextNormalizer, normalize_dataset

# Paths
VOCAB_ANALYSIS_JSON = './outputs/vocab_analysis/vocabulary_analysis.json'
OUTPUT_DIR = '../preprocessed_data/'

print("="*80)
print("TEXT NORMALIZATION PIPELINE")
print("="*80)

# Normalize training data
print("\n[1/2] Normalizing training data...")
train_normalized, normalizer = normalize_dataset(
    input_path=TRAIN_PATH,
    output_path=f'{OUTPUT_DIR}train_normalized.csv',
    vocab_analysis_path=VOCAB_ANALYSIS_JSON,
    normalize_prices=False,  # Keep prices for now (optional)
    normalize_percentages=False,  # Keep percentages for now (optional)
    text_column='text',
    output_column='text_normalized'
)

# Normalize test data (use same normalizer for consistency)
print("\n[2/2] Normalizing test data...")
try:
    test_df = pd.read_csv(TEST_PATH)
    test_normalized = normalizer.normalize_dataframe(
        test_df,
        text_column='text',
        output_column='text_normalized',
        inplace=False
    )
    test_normalized.to_csv(f'{OUTPUT_DIR}test_normalized.csv', index=False)
    print(f"✓ Normalized test dataset saved to {OUTPUT_DIR}test_normalized.csv")
except Exception as e:
    print(f"⚠ Could not normalize test data: {e}")

# Save normalizer configuration for reproducibility
normalizer.save_config(f'{OUTPUT_DIR}normalizer_config.json')

print("\n" + "="*80)
print("✓ NORMALIZATION COMPLETE")
print("="*80)
print(f"\nOutput files:")
print(f"  • {OUTPUT_DIR}train_normalized.csv")
print(f"  • {OUTPUT_DIR}test_normalized.csv")
print(f"  • {OUTPUT_DIR}normalizer_config.json")
print("="*80)

### Validation: Sample Comparisons

Let's examine how the normalization transformed actual tweets:

In [ ]:
# SAMPLE VALIDATION
# Compare original vs normalized text to verify transformations

print("="*80)
print("SAMPLE COMPARISONS: ORIGINAL vs NORMALIZED")
print("="*80)

# Get samples where changes were made
comparisons = normalizer.get_sample_comparisons(
    train_normalized,
    n_samples=15,
    text_column='text',
    normalized_column='text_normalized',
    seed=42
)

for i, (original, normalized) in enumerate(comparisons, 1):
    print(f"\n{'─'*80}")
    print(f"[Sample {i}]")
    print(f"{'─'*80}")
    print(f"ORIGINAL:")
    print(f"  {original}")
    print(f"\nNORMALIZED:")
    print(f"  {normalized}")
    
    # Highlight what changed
    changes = []
    if '$' in original and '[TICKER]' in normalized:
        changes.append('✓ Ticker replaced')
    if any(company.lower() in original.lower() for company in normalizer.companies[:10]):
        if '[COMPANY]' in normalized:
            changes.append('✓ Company replaced')
    if '@' in original and '[USER]' in normalized:
        changes.append('✓ Mention replaced')
    if 'http' in original and '[URL]' in normalized:
        changes.append('✓ URL replaced')
    
    if changes:
        print(f"\nCHANGES: {', '.join(changes)}")

print(f"\n{'='*80}")
print("VALIDATION COMPLETE")
print("="*80)

### Statistical Analysis

Analyze the impact of normalization on vocabulary and class distribution:

In [ ]:
# STATISTICAL IMPACT ANALYSIS

print("="*80)
print("NORMALIZATION IMPACT ANALYSIS")
print("="*80)

# 1. Vocabulary size comparison
print("\n[1] VOCABULARY SIZE")
print("-"*80)

from collections import Counter

def get_vocab_size(texts):
    """Count unique tokens in texts"""
    vocab = Counter()
    for text in texts:
        if pd.notna(text):
            tokens = str(text).lower().split()
            vocab.update(tokens)
    return len(vocab), vocab

original_vocab_size, original_vocab = get_vocab_size(train_normalized['text'])
normalized_vocab_size, normalized_vocab = get_vocab_size(train_normalized['text_normalized'])

vocab_reduction = original_vocab_size - normalized_vocab_size
reduction_pct = (vocab_reduction / original_vocab_size) * 100

print(f"Original vocabulary:    {original_vocab_size:,} unique tokens")
print(f"Normalized vocabulary:  {normalized_vocab_size:,} unique tokens")
print(f"Reduction:              {vocab_reduction:,} tokens ({reduction_pct:.2f}%)")

# 2. Token frequency analysis
print("\n[2] NEW SPECIAL TOKENS")
print("-"*80)

special_tokens = ['[TICKER]', '[COMPANY]', '[USER]', '[URL]', '[PRICE]', '[PERCENT]']
print(f"{'Token':<15} {'Frequency':<15} {'% of corpus':<15}")
print("-"*80)

total_tokens = sum(normalized_vocab.values())
for token in special_tokens:
    count = normalized_vocab.get(token.lower(), 0)
    if count > 0:
        pct = (count / total_tokens) * 100
        print(f"{token:<15} {count:<15,} {pct:<15.3f}%")

# 3. Text length comparison
print("\n[3] TEXT LENGTH STATISTICS")
print("-"*80)

train_normalized['original_length'] = train_normalized['text'].str.len()
train_normalized['normalized_length'] = train_normalized['text_normalized'].str.len()

print(f"{'Metric':<25} {'Original':<15} {'Normalized':<15} {'Change':<15}")
print("-"*80)

orig_mean = train_normalized['original_length'].mean()
norm_mean = train_normalized['normalized_length'].mean()
print(f"{'Mean length (chars)':<25} {orig_mean:<15.2f} {norm_mean:<15.2f} {norm_mean - orig_mean:<15.2f}")

orig_median = train_normalized['original_length'].median()
norm_median = train_normalized['normalized_length'].median()
print(f"{'Median length (chars)':<25} {orig_median:<15.2f} {norm_median:<15.2f} {norm_median - orig_median:<15.2f}")

# 4. Class distribution (should be unchanged)
print("\n[4] CLASS DISTRIBUTION (SANITY CHECK)")
print("-"*80)

label_counts = train_normalized['label'].value_counts().sort_index()
print(f"{'Class':<15} {'Count':<15} {'Percentage':<15}")
print("-"*80)
for label, count in label_counts.items():
    pct = (count / len(train_normalized)) * 100
    print(f"{label_map[label]:<15} {count:<15,} {pct:<15.2f}%")

# 5. Change rate
print("\n[5] NORMALIZATION CHANGE RATE")
print("-"*80)

changed_samples = (train_normalized['text'] != train_normalized['text_normalized']).sum()
change_rate = (changed_samples / len(train_normalized)) * 100

print(f"Samples changed:    {changed_samples:,} / {len(train_normalized):,}")
print(f"Change rate:        {change_rate:.2f}%")
print(f"Unchanged samples:  {len(train_normalized) - changed_samples:,} ({100-change_rate:.2f}%)")

print("\n" + "="*80)
print("✓ ANALYSIS COMPLETE")
print("="*80)

# Summary
print("\n📊 KEY FINDINGS:")
print(f"  ✓ Vocabulary reduced by {vocab_reduction:,} tokens ({reduction_pct:.2f}%)")
print(f"  ✓ {changed_samples:,} samples normalized ({change_rate:.2f}%)")
print(f"  ✓ Special tokens added: {sum(1 for t in special_tokens if normalized_vocab.get(t.lower(), 0) > 0)}")
print(f"  ✓ Class distribution preserved: ✓")
print("\n💡 Expected accuracy improvement: +3-6% (from vocabulary reduction + better generalization)")
print("="*80)